# RuSearchRank, этап 1A — полный BM25-поиск по MIRACL-RU

Этот notebook служит компактным сценарием запуска `rusearchrank.cli` в Linux/Colab. Он скачивает официальный сжатый индекс Lucene размером 6,42 GiB, строит top-100 для обучающей выборки и официальный top-1000 для dev, оценивает неизменённую выдачу dev официальным NIST `trec_eval`, затем выполняет короткую, но полноценную проверку корпуса. После этого создаются отдельная зафиксированная выдача top-100 для dev и кеш кандидатов проекта. Тексты документов берутся из официальных статических сегментов `miracl-corpus-v1.0-ru/docs-0.jsonl.gz` … `docs-19.jsonl.gz` в зафиксированной ревизии; устаревший сценарий набора данных `miracl-corpus.py` не запускается, а `trust_remote_code` не используется. Результат сохраняется в `artifacts/rusearchrank_phase1_results.zip`. Зависимостям Python и распакованному индексу требуется дополнительное временное место; перед запуском должно быть свободно не менее 30 GiB.

Индекс Lucene, кеши корпуса, Hugging Face и Pyserini, окружение Python, каталог `.git` и временные рабочие файлы не входят в ZIP и не должны добавляться в Git. Архив содержит только файлы кандидатов Parquet, top-100 обучающей выборки, исходный top-1000 dev, производный top-100 dev и три файла проверки этапа 1 в формате JSON. Выполните все 15 ячеек по порядку сверху вниз; дополнительные ячейки не нужны. Ресурсоёмкие операции разделены: ячейка 8 строит выдачу обучающей выборки, ячейка 9 — выдачу dev, ячейка 10 выполняет обязательное официальное оценивание, ячейка 11 проверяет настоящий корпус, ячейка 12 строит кеш кандидатов, а ячейка 14 упаковывает результаты.

Существующие корректные файлы `artifacts/runs/*.trec` из предыдущего сеанса используются без изменений: поиск не запускается повторно, а исходные выдачи не перезаписываются, пока в ячейке 3 параметру `ALLOW_OVERWRITE_RUNS_AND_CACHE` явно не присвоено значение `True`.

In [ ]:
# Cell 2 — fail-fast Linux, hardware, and disk gate (no downloads).
import os, platform, shutil, sys
from pathlib import Path

MIN_FREE_GIB = 30
disk = shutil.disk_usage('/content' if Path('/content').is_dir() else '.')
mem_kib = next((int(line.split()[1]) for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')), 0) if Path('/proc/meminfo').is_file() else 0
environment = {
    'os': platform.platform(),
    'python': platform.python_version(),
    'cpu': platform.processor() or platform.machine(),
    'cpu_count': os.cpu_count(),
    'ram_gib': round(mem_kib / 1024**2, 2),
    'disk_free_gib': round(disk.free / 1024**3, 2),
}
print(environment)
if platform.system() != 'Linux':
    raise RuntimeError('This runner must execute on Linux (Google Colab is supported).')
if disk.free < MIN_FREE_GIB * 1024**3:
    raise RuntimeError(f'Insufficient free disk: {environment["disk_free_gib"]} GiB; at least {MIN_FREE_GIB} GiB is required before downloading the index.')

In [ ]:
# Cell 3 — one diagnostic command helper; clone/fast-forward the exact branch.
import json, os, shlex, subprocess, sys, threading
from pathlib import Path

LOG_DIR = Path('/content/rusearchrank-logs')

def run_checked(command, *, cwd=None, env=None, stream=False, log_path=None, stage=None):
    """Run one command, always showing the command, cwd, key env, both pipes and the return code."""
    command = [str(part) for part in command]
    effective_cwd = Path(cwd or Path.cwd()).resolve()
    effective_env = os.environ.copy() if env is None else env.copy()
    shown_env = {name: effective_env.get(name) for name in ('JAVA_HOME', 'PATH', 'PYTHONPATH', 'HF_HOME', 'HF_HUB_CACHE') if effective_env.get(name)}
    shown_env['PATH'] = (shown_env.get('PATH') or '')[:200] + ('…' if len(shown_env.get('PATH') or '') > 200 else '')
    print('+', shlex.join(command), flush=True)
    print('  cwd:', effective_cwd, flush=True)
    print('  env:', json.dumps(shown_env, ensure_ascii=False), flush=True)
    try:
        if stream:
            process = subprocess.Popen(command, cwd=effective_cwd, env=effective_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
            stdout_lines, stderr_lines = [], []
            def pump(pipe, destination, collected):
                for line in iter(pipe.readline, ''):
                    collected.append(line)
                    print(line, end='', file=destination, flush=True)
                pipe.close()
            workers = [threading.Thread(target=pump, args=(process.stdout, sys.stdout, stdout_lines), daemon=True), threading.Thread(target=pump, args=(process.stderr, sys.stderr, stderr_lines), daemon=True)]
            for worker in workers: worker.start()
            returncode = process.wait()
            for worker in workers: worker.join()
            stdout, stderr = ''.join(stdout_lines), ''.join(stderr_lines)
        else:
            result = subprocess.run(command, cwd=effective_cwd, env=effective_env, text=True, capture_output=True, check=False)
            returncode, stdout, stderr = result.returncode, result.stdout, result.stderr
            if stdout: print(stdout, end='' if stdout.endswith('\n') else '\n')
            if stderr: print(stderr, end='' if stderr.endswith('\n') else '\n', file=sys.stderr)
    except OSError as exc:
        returncode, stdout, stderr = None, '', f'{type(exc).__name__}: {exc}'
        print('stdout:', stdout, file=sys.stderr)
        print('stderr:', stderr, file=sys.stderr)
    print('  return code:', returncode, flush=True)
    record = {'stage': stage, 'command': command, 'cwd': str(effective_cwd), 'env': shown_env, 'returncode': returncode, 'stdout': stdout, 'stderr': stderr}
    log_path = Path(log_path) if log_path else (LOG_DIR / (str(stage or 'command') + '.json'))
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(json.dumps(record, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('  complete log:', log_path, flush=True)
    if returncode != 0:
        tail = ''.join((stderr or stdout or '').splitlines(keepends=True)[-25:])
        raise RuntimeError(
            f'stage={stage or "command"} failed with return code {returncode}\n'
            f'command: {shlex.join(command)}\ncwd: {effective_cwd}\n'
            f'complete log: {log_path}\nlast output lines:\n{tail}'
        )
    return record

REPO_URL = 'https://github.com/kopanevk/ru-search-rank.git'
BRANCH = 'phase-0'
REPO_DIR = Path('/content/ru-search-rank')
ALLOW_OVERWRITE_RUNS_AND_CACHE = False  # Set True only for an intentional rerun.
if (REPO_DIR / '.git').is_dir():
    run_checked(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, stage='git_fetch')
    run_checked(['git', 'checkout', BRANCH], cwd=REPO_DIR, stage='git_checkout')
    run_checked(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, stage='git_pull')
else:
    run_checked(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], cwd='/content', stage='git_clone')
os.chdir(REPO_DIR)
run_checked(['git', 'log', '-1', '--oneline', '--decorate'], cwd=REPO_DIR, stage='git_head')

In [ ]:
# Cell 4 — install Java 21 and pinned official NIST trec_eval v9.0.8.
import glob, os, re, subprocess
from pathlib import Path

run_checked(['apt-get', 'update', '-qq'], stage='apt_update')
run_checked(['apt-get', 'install', '-y', '-qq', 'openjdk-21-jdk-headless', 'build-essential'], stage='apt_java')
java_candidates = sorted(glob.glob('/usr/lib/jvm/java-21*/bin/java'))
if not java_candidates:
    raise RuntimeError('OpenJDK 21 was installed but its java executable was not found.')
java_bin = Path(java_candidates[0]).resolve()
JAVA_HOME = java_bin.parent.parent
os.environ['JAVA_HOME'] = str(JAVA_HOME)
os.environ['PATH'] = f'{java_bin.parent}:' + os.environ['PATH']
java_probe = run_checked([str(java_bin), '-version'], env=os.environ, stage='java_version')
java_version = java_probe['stderr'] or java_probe['stdout']
print(java_version)
if not re.search(r'(?:version\s+"|openjdk\s+)21(?:[."]|$)', java_version.lower()):
    raise RuntimeError(f'Active Java is not 21; JAVA_HOME={JAVA_HOME}')

TREC_EVAL_TAG = 'v9.0.8'
TREC_EVAL_DIR = Path('/content/trec_eval-v9.0.8')
if not (TREC_EVAL_DIR / '.git').is_dir():
    run_checked(['git', 'clone', '--depth', '1', '--branch', TREC_EVAL_TAG, 'https://github.com/usnistgov/trec_eval.git', str(TREC_EVAL_DIR)], cwd='/content', stage='trec_eval_clone')
else:
    run_checked(['git', 'fetch', '--tags', 'origin'], cwd=TREC_EVAL_DIR, stage='trec_eval_fetch')
    run_checked(['git', 'checkout', '--detach', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR, stage='trec_eval_checkout')
trec_head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=TREC_EVAL_DIR, stage='trec_eval_head')['stdout'].strip()
trec_tag_commit = run_checked(['git', 'rev-list', '-n', '1', TREC_EVAL_TAG], cwd=TREC_EVAL_DIR, stage='trec_eval_tag')['stdout'].strip()
if trec_head != trec_tag_commit:
    raise RuntimeError(f'trec_eval checkout is not exactly {TREC_EVAL_TAG}: {trec_head}')
run_checked(['make', '-j', str(os.cpu_count() or 2)], cwd=TREC_EVAL_DIR, stage='trec_eval_make')
run_checked(['install', '-m', '0755', str(TREC_EVAL_DIR / 'trec_eval'), '/usr/local/bin/trec_eval'], stage='trec_eval_install')
trec_eval_probe = run_checked(['/usr/local/bin/trec_eval', '-h'], stage='trec_eval_probe')
print('trec_eval expected version', TREC_EVAL_TAG, trec_eval_probe['stdout'] or trec_eval_probe['stderr'])

In [ ]:
# Cell 5 — create an isolated Python 3.12 environment without replacing system Python.
import shutil, subprocess, sys
from pathlib import Path

VENV_DIR = Path('/content/rusearchrank-py312')
if sys.version_info[:2] == (3, 12):
    run_checked(['apt-get', 'install', '-y', '-qq', 'python3.12-venv'], stage='apt_venv')
    python312 = Path(sys.executable)
    if not (VENV_DIR / 'bin/python').is_file():
        run_checked([str(python312), '-m', 'venv', str(VENV_DIR)], stage='create_venv')
else:
    UV_VERSION = '0.8.13'
    uv_prefix = Path('/content/uv-bootstrap')
    uv = uv_prefix / 'bin/uv'
    if not uv.is_file():
        run_checked([sys.executable, '-m', 'pip', 'install', '--prefix', str(uv_prefix), f'uv=={UV_VERSION}'], stage='install_uv')
    run_checked([str(uv), 'python', 'install', '3.12'], stage='uv_python')
    if not (VENV_DIR / 'bin/python').is_file():
        run_checked([str(uv), 'venv', '--python', '3.12', str(VENV_DIR)], stage='uv_venv')
RUN_PYTHON = VENV_DIR / 'bin/python'
python_probe = run_checked([str(RUN_PYTHON), '--version'], stage='venv_python_version')
actual_python = (python_probe['stdout'] or python_probe['stderr']).strip()
print(actual_python, RUN_PYTHON)
if not actual_python.startswith('Python 3.12.'):
    raise RuntimeError(f'Isolated interpreter is not Python 3.12: {actual_python}')

In [ ]:
# Cell 6 — install the project and pinned retrieval dependency, then test under Python 3.12.
import os, subprocess

run_checked([str(RUN_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip'], cwd=REPO_DIR, stage='pip_upgrade')
run_checked([str(RUN_PYTHON), '-m', 'pip', 'install', '-e', f'{REPO_DIR}[retrieval]'], cwd=REPO_DIR, stage='pip_install_project')
# Record the exact resolved versions the compatibility matrix in reports/audit depends on.
version_probe = (
    "import importlib.metadata as m, sys;"
    "print(sys.version);"
    "print({name: m.version(name) for name in ('pyserini', 'datasets', 'huggingface-hub', 'pandas', 'pyarrow', 'numpy')})"
)
run_checked([str(RUN_PYTHON), '-c', version_probe], cwd=REPO_DIR, env=os.environ, stage='version_probe')
# Exact isolated-environment equivalent of: python -m pytest -q
run_checked([str(RUN_PYTHON), '-m', 'pytest', '-q'], cwd=REPO_DIR, env=os.environ, stage='pytest')
run_checked([str(RUN_PYTHON), 'scripts/validate_phase1_notebook.py'], cwd=REPO_DIR, env=os.environ, stage='validate_notebook')

In [ ]:
# Cell 7 — download official topics/qrels, then run full retrieval preflight/index smoke.
import os, re, subprocess

CONFIG = 'configs/retrieval.yaml'
def cli(*arguments, stream=False):
    command = [str(RUN_PYTHON), '-m', 'rusearchrank.cli', *arguments]
    stage = re.sub(r'[^A-Za-z0-9_.-]+', '_', '__'.join(map(str, arguments)))
    log_path = REPO_DIR / 'artifacts/work/phase1/notebook_logs' / (stage + '.json')
    return run_checked(command, cwd=REPO_DIR, env=os.environ, stream=stream, log_path=log_path, stage=stage)

cli('prepare-annotations', '--config', CONFIG)
cli('preflight', '--config', CONFIG, '--stage', 'retrieval', '--check-index', stream=True)

In [ ]:
# Cell 8 — train BM25 top-100 from the validated local official TSV (4,683 queries).
# An existing valid artifacts/runs/train_bm25_top100.trec is revalidated and reused, never rerun.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('run-bm25', '--config', CONFIG, '--split', 'train', *overwrite, stream=True)

In [ ]:
# Cell 9 — official dev BM25 with --hits 1000; preserve the raw run for reproduction.
# An existing valid artifacts/runs/dev_bm25_top1000.trec is revalidated and reused, never rerun.
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('run-bm25', '--config', CONFIG, '--split', 'dev', *overwrite, stream=True)

In [ ]:
# Cell 10 — evaluate the untouched dev top-1000 with official NIST trec_eval commands.
# run_checked stops here with command/stdout/stderr/return code unless the gate passes
# (nDCG@10 0.334 ± 0.002 and Recall@100 0.661 ± 0.005). No candidate cache is built otherwise.
cli('evaluate-bm25', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 11 — REAL COLAB SMOKE: prove real corpus access before the full candidate cache.
# Downloads exactly one official static shard (miracl-corpus-v1.0-ru/docs-0.jsonl.gz) from the
# pinned immutable revision, parses real gzip/JSON, filters real passages, writes and re-reads a
# real Parquet, builds a real manifest and ZIP, extracts it and re-verifies the SHA-256.
# It never downloads the whole corpus, never runs BM25 and never writes a production artifact.
smoke = cli('smoke-corpus-access', '--config', CONFIG, '--max-rows', '2000', '--min-passages', '25', stream=True)
import json as _json
_lines = smoke['stdout'].splitlines()
_start = max(index for index, line in enumerate(_lines) if line.rstrip() == '{')
report = _json.loads('\n'.join(_lines[_start:]))
CORPUS_SMOKE_PASSED = report['status'] == 'PASS' and not report['full_corpus_downloaded']
print({check['check']: check['status'] for check in report['checks']})
if not CORPUS_SMOKE_PASSED:
    raise RuntimeError('Corpus smoke did not pass; do not run Cell 12 until it does.')
print('REAL COLAB SMOKE PASSED — Cell 12 may run.')

In [ ]:
# Cell 12 — preflight, idempotent stable dev top-100, three-state cache, passages, qrels audit.
# Requires the Cell 11 smoke; passages stream from the pinned static shards in batches.
if not globals().get('CORPUS_SMOKE_PASSED'):
    raise RuntimeError('Run Cell 11 (REAL COLAB SMOKE) before building the candidate cache.')
overwrite = ['--overwrite'] if ALLOW_OVERWRITE_RUNS_AND_CACHE else []
cli('preflight', '--config', CONFIG, '--stage', 'candidate-cache')
cli('build-candidate-cache', '--config', CONFIG, *overwrite, stream=True)
cli('audit-qrels', '--config', CONFIG, '--overwrite')

In [ ]:
# Cell 13 — candidate, query, passage, three-state, top-K, and stable-order validation.
cli('validate-candidates', 'artifacts/candidates/train_top100.parquet', '--config', CONFIG)
cli('validate-candidates', 'artifacts/candidates/dev_top100.parquet', '--config', CONFIG)
cli('preflight', '--config', CONFIG, '--stage', 'package')

In [ ]:
# Cell 14 — package the explicit portable allowlist; caches/index/environment are excluded.
# --overwrite replaces the placeholder manifest that ships in the repository.
cli('package-phase1', '--config', CONFIG, '--overwrite', stream=True)

In [ ]:
# Cell 15 — show ZIP size, SHA-256, exact contents, then download (optional Drive copy).
import hashlib, shutil, zipfile
from pathlib import Path

archive_path = REPO_DIR / 'artifacts/rusearchrank_phase1_results.zip'
digest = hashlib.sha256()
with archive_path.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
with zipfile.ZipFile(archive_path) as archive:
    contents = archive.namelist()
print({'path': str(archive_path), 'size_bytes': archive_path.stat().st_size, 'sha256': digest.hexdigest(), 'contents': contents})

DRIVE_DESTINATION = ''  # Optional, e.g. '/content/drive/MyDrive/rusearchrank_phase1_results.zip'.
if DRIVE_DESTINATION:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy2(archive_path, DRIVE_DESTINATION)
    print('Copied to', DRIVE_DESTINATION)
try:
    from google.colab import files
except ImportError:
    print('Not running in Colab; retrieve the validated ZIP from', archive_path)
else:
    files.download(str(archive_path))